# Training YOLO26n untuk Deteksi Obat — Kaggle Notebook (GPU Gratis)

Notebook ini untuk **melatih model deteksi obat** memakai GPU gratis Kaggle (T4 x2 atau P100),
bukan di laptop Lenovo V14 G2 ITL (MX350 2GB terlalu kecil dan lambat untuk training —
lihat dokumen desain bagian 3, 4, dan 9). Setelah training selesai, file `best.onnx` yang
dihasilkan di-download dan dipakai di laptop untuk **inference saja** (jauh lebih ringan
daripada training).

## Cara pakai notebook ini di Kaggle
1. Buka https://www.kaggle.com/code -> **New Notebook**, lalu paste/upload file ini.
2. Klik **Settings** (panel kanan) -> **Accelerator** -> pilih **GPU T4 x2** (atau P100).
3. **Settings -> Internet -> On** (dibutuhkan untuk `pip install ultralytics` dan download dataset).
4. Siapkan dataset (lihat sel "Dataset" di bawah) — via `+ Add Data` di Kaggle, atau `kagglehub`.
5. Jalankan semua sel dari atas ke bawah (Run All).
6. Di akhir, download `medicine_model_export.zip` dari panel Output Kaggle.

Kuota GPU gratis Kaggle: **~30 jam/minggu**, reset setiap Sabtu. Training YOLO26n nano
pada beberapa ribu gambar dengan 120 epoch biasanya selesai dalam 1-3 jam di T4 —
jauh di bawah kuota mingguan.


## 1. Cek GPU aktif
Pastikan ini menunjukkan Tesla T4 (atau P100) sebelum lanjut — kalau tidak muncul,
training akan jalan di CPU dan sangat lambat.


In [ ]:
!nvidia-smi


## 2. Install Ultralytics (YOLO11)


In [ ]:
!pip install -q ultralytics onnx onnxslim
import ultralytics
ultralytics.checks()


## 3. Dataset

**4 dataset Roboflow sudah dikumpulkan, semua format `yolo26` (Anda sudah punya file aslinya):**

| Dataset (file asli) | Kelas | Gambar | Ikut digabung? |
|---|---|---|---|
| `Pharmacy.v7i.yolo26.zip` | 44 SKU nyata (alphain, fenaxon, sarita, dst) | 1454 | Ya |
| `pharmacy-shelf-detect.v1i.yolo26.zip` | 4 SKU nyata (Panadol_coldflu, alphintern, augmentin_1gm, panadol_extra) | 411 | Ya |
| `pharmacy robot.v2i.yolo26.zip` | 4 SKU nyata (panadol, paymol, revanin, riva-n) | 414 | Ya |
| `Pharmacy.v3i.yolo26.zip` | 10 kelas berlabel ANGKA `1`-`10` saja | 1538 | **TIDAK** (lihat catatan) |

**Catatan tentang `Pharmacy.v7i.yolo26.zip`:** ini re-export format `yolo26` dari proyek Roboflow
yang sama (`sito-s02gz/pharmacy-bk2qv`, versi 7) yang sebelumnya diverifikasi lewat export
`yolov8`. Sudah dicek ulang -- 44 kelas sama persis, 1454 gambar sama persis, dan setelah
konversi poligon->bbox hasil labelnya **identik byte-per-byte** dengan export `yolov8` yang lama
(864 bbox asli + 6039 dari poligon, 0 baris rusak -- sama persis kedua export). Jadi tidak ada
bedanya secara isi data, cuma nama file yang berubah.

**Kenapa `Pharmacy.v3i.yolo26.zip` dikeluarkan:** nama kelasnya cuma angka (`1`..`10`), tidak
membawa identitas SKU apa pun -- ini dataset yang sama yang sudah diperiksa sebelumnya (proyek
`pharmacy-jl5t1`), cuma diekspor ulang dalam format yolo26. Kalau Anda tahu angka `1`-`10` itu
sebenarnya SKU apa (mis. dari urutan pelabelan aslinya), beri tahu saya dan ini bisa dimasukkan
juga -- untuk sekarang sel di bawah otomatis MENDETEKSI & MELEWATI dataset semacam ini (kelas yang
seluruhnya berupa angka) supaya tidak diam-diam masuk sebagai SKU palsu.

**3 dataset yang digabung -> total 52 kelas** (union nama kelas, sudah diverifikasi lokal: 8182
objek, 2279 gambar, 0 kesalahan index -- lihat `tests/test_merge_datasets.py`). Tiap dataset
sumber punya index kelasnya sendiri-sendiri, jadi sel di bawah menulis ulang index tiap label ke
daftar kelas gabungan (bukan sekadar menyalin folder) -- kalau ini dilewati, model akan belajar
label yang SALAH.

**3 kemiripan nama lintas-dataset yang perlu Anda putuskan (TIDAK digabung otomatis, sengaja
dibiarkan sebagai kelas terpisah sampai Anda konfirmasi):**
- `alphain` (v7) vs `alphintern` (shelf-detect) -- produk yang sama atau berbeda?
- `panadol` (robot) vs `Panadol_coldflu` (shelf-detect) -- varian umum vs spesifik, atau salah label?
- `panadol` (robot) vs `panadol_extra` (shelf-detect) -- sama.

**Catatan lain:** dataset `pharmacy robot.v2i` cuma py folder `train` (tidak ada split
valid/test dari sumbernya), jadi 4 kelasnya (panadol, paymol, revanin, riva-n) TIDAK akan pernah
muncul di validasi -- mAP untuk keempat kelas itu tidak akan ke-report di sel 6, hanya
ikut training. Roboflow juga sempat menerapkan augmentasi (flip/crop/rotasi/brightness/blur) pada
dataset ini sebelum ekspor, jadi sebagian dari 414 gambarnya adalah variasi dari foto asli yang sama.

**Soal error "Conflicting files detected ... data.yaml" saat upload ke Kaggle:** ini bug dari
tool export Roboflow -- tiap zip (termasuk `Pharmacy.v7i.yolo26.zip`) menulis 3 salinan
`data.yaml` di path internal yang sama persis. Sudah dicek byte-per-byte: isinya identik, jadi
aman menerima resolusi apapun yang ditawarkan dialog konflik Kaggle (keep one / overwrite) --
tidak ada data yang hilang.

**Cara pakai:** di Kaggle, **+ Add Input -> Upload** untuk KETIGA file asli (`Pharmacy.v7i.yolo26.zip`,
`pharmacy-shelf-detect.v1i.yolo26.zip`, `pharmacy robot.v2i.yolo26.zip` -- boleh juga upload
`Pharmacy.v3i.yolo26.zip`, sel di bawah akan mendeteksi & melewatinya otomatis), lalu jalankan sel
Opsi C di bawah.


In [ ]:
# --- Opsi A: download dataset publik Kaggle lewat kagglehub ---
# import kagglehub
# path = kagglehub.dataset_download("trainingdatapro/pills-detection-dataset")

# --- Opsi B: dataset dari Roboflow (paste snippet yang diberikan Roboflow saat export) ---
# !pip install -q roboflow
# from roboflow import Roboflow
# rf = Roboflow(api_key="API_KEY_ANDA")

# --- Opsi C (DIREKOMENDASIKAN): gabungkan semua dataset Pharmacy yang sudah diupload ---
# Logikanya sama persis dengan scripts/convert_roboflow_dataset.py + scripts/merge_datasets.py
# yang sudah diuji di proyek (lihat tests/test_roboflow_converter.py, tests/test_merge_datasets.py).
# Ditulis ulang di sini karena Kaggle notebook berdiri sendiri, tidak akses repo Anda.
import glob, pathlib, shutil, yaml, difflib, zipfile

SPLITS = ("train", "valid", "test")


def convert_label_line(line):
    parts = line.strip().split()
    if not parts:
        return None
    cls, coords = parts[0], [float(x) for x in parts[1:]]
    if len(coords) == 4:
        return cls, coords
    if len(coords) < 6 or len(coords) % 2 != 0:
        return None
    xs, ys = coords[0::2], coords[1::2]
    x_min, x_max, y_min, y_max = min(xs), max(xs), min(ys), max(ys)
    return cls, [(x_min + x_max) / 2, (y_min + y_max) / 2, x_max - x_min, y_max - y_min]


def convert_dataset(raw_root, out_root):
    """Poligon -> bbox (baris 4-token disalin apa adanya). Return (n_bbox, n_poly, n_bad)."""
    n_bbox = n_poly = n_bad = 0
    for split in SPLITS:
        src_img, src_lbl = raw_root / split / "images", raw_root / split / "labels"
        if not src_img.exists():
            continue
        dst_img, dst_lbl = out_root / split / "images", out_root / split / "labels"
        dst_img.mkdir(parents=True, exist_ok=True)
        dst_lbl.mkdir(parents=True, exist_ok=True)
        for p in src_img.iterdir():
            shutil.copy(p, dst_img / p.name)
        for lbl_path in src_lbl.glob("*.txt"):
            out_lines = []
            for line in lbl_path.read_text().splitlines():
                if not line.strip():
                    continue
                n_tok = len(line.strip().split()) - 1
                result = convert_label_line(line)
                if result is None:
                    n_bad += 1
                    continue
                cls, (cx, cy, w, h) = result
                out_lines.append(f"{cls} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")
                if n_tok == 4:
                    n_bbox += 1
                else:
                    n_poly += 1
            (dst_lbl / lbl_path.name).write_text("\n".join(out_lines) + ("\n" if out_lines else ""))
    return n_bbox, n_poly, n_bad


def find_data_yaml_roots(base):
    return {pathlib.Path(p).parent for p in glob.glob(f"{base}/**/data.yaml", recursive=True)}


# 1) Temukan semua dataset yang di-upload, konversi tiap satu (poligon -> bbox)
#
# Kaggle punya 2 mode untuk file yang di-upload sebagai dataset: kadang otomatis diekstrak
# jadi folder (data.yaml langsung kebaca di /kaggle/input/<slug>/data.yaml), tapi untuk
# dataset besar/tertentu kadang dibiarkan sebagai file .zip mentah di /kaggle/input/<slug>/.
# Sel ini menangani KEDUANYA -- kalau tidak ketemu data.yaml langsung, ekstrak dulu zip
# mentah yang ada, baru cari lagi.
raw_roots = find_data_yaml_roots("/kaggle/input")

# Ekstrak SEMUA file .zip mentah yang ditemukan -- bukan cuma kalau raw_roots masih kosong,
# supaya dataset yang sudah-diekstrak-otomatis dan yang-masih-berupa-zip-mentah bisa bercampur
# (mis. 2 dataset sudah otomatis diekstrak Kaggle, 1 dataset lain masih zip mentah) tanpa yang
# masih-zip itu diam-diam terlewat.
already_covered_dirs = set(raw_roots)  # jangan ekstrak ulang zip yang folder-nya SUDAH punya data.yaml
zips = [
    zp for zp in glob.glob("/kaggle/input/**/*.zip", recursive=True)
    if pathlib.Path(zp).parent not in already_covered_dirs
]
if zips:
    print(f"Ditemukan {len(zips)} file .zip mentah di /kaggle/input/ (belum diekstrak Kaggle), mengekstrak dulu...")
    for zp in zips:
        zp = pathlib.Path(zp)
        dest = pathlib.Path("/kaggle/working") / f"{zp.parent.name}__{zp.stem}_raw"
        if not dest.exists():
            print(f"  [EKSTRAK] {zp} -> {dest}")
            with zipfile.ZipFile(zp) as zf:
                zf.extractall(dest)
        raw_roots |= find_data_yaml_roots(str(dest))

if not raw_roots:
    print("=== Isi /kaggle/input/ saat ini (untuk diagnosa) ===")
    entries = sorted(glob.glob("/kaggle/input/*"))
    if not entries:
        print("  (KOSONG -- tidak ada dataset yang ter-attach ke NOTEBOOK ini sama sekali)")
    for p in entries:
        print(" ", p)
        for sub in sorted(glob.glob(p + "/*"))[:10]:
            print("     -", sub)
    raise FileNotFoundError(
        "Tidak ditemukan data.yaml maupun file .zip dataset di /kaggle/input/. Penyebab paling umum: "
        "meng-upload file ke Kaggle Datasets TIDAK otomatis menyambungkannya ke notebook ini -- keduanya "
        "langkah terpisah. Perbaikan: buka panel kanan notebook -> '+ Add Input' -> cari dataset Pharmacy "
        "yang sudah Anda upload (Your Datasets / Private) -> klik tombol '+' / 'Add' pada tiap dataset "
        "(Pharmacy v7, pharmacy-shelf-detect v1, pharmacy robot v2) -> tunggu sampai muncul di panel "
        "Input sebelah kanan -> baru jalankan ulang sel ini (Run cell ini saja, tidak perlu Run All lagi)."
    )

raw_roots = sorted(raw_roots)
print(f"Ditemukan {len(raw_roots)} dataset ter-attach: {[r.name for r in raw_roots]}\n")

converted = {}  # nama -> (path, names)
for raw_root in raw_roots:
    cfg = yaml.safe_load(open(raw_root / "data.yaml"))
    names = list(cfg["names"])
    if all(str(n).strip().isdigit() for n in names):
        print(f"[LEWATI] {raw_root.name}: kelasnya cuma angka {names} -- tidak ada identitas SKU, tidak digabung.")
        print("          (kalau Anda tahu angka ini sebenarnya SKU apa, beri tahu saya untuk dimasukkan manual.)")
        continue
    out_root = pathlib.Path(f"/kaggle/working/{raw_root.name}_converted")
    n_bbox, n_poly, n_bad = convert_dataset(raw_root, out_root)
    print(f"[OK] {raw_root.name}: {len(names)} kelas, {n_bbox} bbox asli + {n_poly} dari poligon, {n_bad} baris rusak")
    converted[raw_root.name] = (out_root, names)

if not converted:
    raise RuntimeError(
        "Dataset ter-attach ditemukan, tapi SEMUANYA dilewati (kelas cuma angka, mis. Pharmacy.v3i). "
        "Tambahkan minimal 1 dataset dengan nama SKU asli (Pharmacy.v7i.yolo26, pharmacy-shelf-detect.v1i, "
        "atau pharmacy robot.v2i) lewat '+ Add Input', lalu jalankan ulang sel ini."
    )

# 2) Gabungkan: union nama kelas + tulis ulang index tiap label ke index gabungan
merged_names = []
origin = {}
for dname, (_, names) in converted.items():
    for n in names:
        if n not in merged_names:
            merged_names.append(n)
            origin[n] = dname
name_to_idx = {n: i for i, n in enumerate(merged_names)}

print("\n=== Peringatan kemiripan nama lintas-dataset (cek manual, TIDAK digabung otomatis) ===")
any_warn = False
for i, a in enumerate(merged_names):
    for b in merged_names[i + 1:]:
        if origin[a] == origin[b]:
            continue
        ratio = difflib.SequenceMatcher(None, a.lower(), b.lower()).ratio()
        if ratio > 0.55 and a.lower() != b.lower():
            any_warn = True
            print(f"  '{a}' ({origin[a]}) vs '{b}' ({origin[b]}) (kemiripan: {ratio:.2f}) -- sama atau beda produk? cek manual.")
if not any_warn:
    print("  (tidak ada)")

MERGED_ROOT = pathlib.Path("/kaggle/working/pharmacy_merged")
if MERGED_ROOT.exists():
    shutil.rmtree(MERGED_ROOT)

for dname, (out_root, names) in converted.items():
    local_to_global = {i: name_to_idx[n] for i, n in enumerate(names)}
    for split in SPLITS:
        src_img, src_lbl = out_root / split / "images", out_root / split / "labels"
        if not src_img.exists():
            continue
        dst_img, dst_lbl = MERGED_ROOT / split / "images", MERGED_ROOT / split / "labels"
        dst_img.mkdir(parents=True, exist_ok=True)
        dst_lbl.mkdir(parents=True, exist_ok=True)
        for p in src_img.iterdir():
            shutil.copy(p, dst_img / f"{dname}__{p.name}")
        for lbl_path in src_lbl.glob("*.txt"):
            out_lines = []
            for line in lbl_path.read_text().splitlines():
                if not line.strip():
                    continue
                parts = line.split()
                global_cls = local_to_global[int(parts[0])]
                out_lines.append(" ".join([str(global_cls)] + parts[1:]))
            (dst_lbl / f"{dname}__{lbl_path.name}").write_text("\n".join(out_lines) + ("\n" if out_lines else ""))

yaml.safe_dump(
    {"path": str(MERGED_ROOT), "train": "train/images", "val": "valid/images", "test": "test/images",
     "nc": len(merged_names), "names": merged_names},
    open(MERGED_ROOT / "data.yaml", "w"), sort_keys=False,
)
print(f"\nTotal kelas gabungan: {len(merged_names)}")
print(f"Dataset gabungan siap di: {MERGED_ROOT}")


## 4. `data.yaml` sudah otomatis ditulis oleh sel di atas

Sel Opsi C sudah menulis `data.yaml` gabungan ke `/kaggle/working/pharmacy_merged/data.yaml`
(52 kelas kalau ketiga dataset ter-upload semua). **Lewati sel di bawah** kecuali Anda pakai
Opsi A/B (dataset lain) alih-alih Opsi C -- dalam hal itu, edit `names` & `path` di sel di bawah
sesuai dataset pilihan Anda.


In [ ]:
# HANYA jalankan sel ini kalau TIDAK pakai Opsi C di atas (mis. pakai dataset Kaggle publik
# dari Opsi A, atau dataset SKU Anda sendiri). Kalau sudah pakai Opsi C, LEWATI sel ini --
# data.yaml gabungan sudah ada di /kaggle/working/pharmacy_merged/data.yaml.

data_yaml_content = '''
path: /kaggle/input/NAMA_DATASET_ANDA
train: images/train
val: images/val

names:
  0: paracetamol_500mg
  1: amoxicillin_250mg
  2: cetirizine_10mg
  # tambahkan SKU lain sesuai dataset Anda
'''

with open("/kaggle/working/data.yaml", "w") as f:
    f.write(data_yaml_content)

print(data_yaml_content)


## 5. Training

`yolo26n.pt` (nano, COCO-pretrained, rilis Ultralytics Januari 2026) dipakai sebagai
starting checkpoint. Sudah diuji langsung terhadap yolo11n.pt di proyek ini (lihat
dokumen desain): decoder ONNX custom kita cocok 5/5 dengan output resmi Ultralytics,
~15% lebih cepat di CPU (40.6ms vs 47.6ms/frame), parameter lebih kecil (2.4M vs 2.6M),
dan mAP COCO sedikit lebih tinggi (40.9 vs 39.5) — cocok untuk deploy balik ke MX350
2GB VRAM setelah training.

`patience=30` menghentikan training lebih awal kalau val mAP sudah plateau, supaya tidak
membuang kuota GPU gratis Kaggle secara sia-sia.


In [ ]:
try:
    from ultralytics import YOLO
except ModuleNotFoundError:
    # Jaga-jaga kalau sel 2 (install) belum dijalankan di sesi ini, atau sesi Kaggle
    # sempat restart -- install ulang di sini supaya sel training ini tidak gagal.
    print("ultralytics belum terinstall di sesi ini, menginstall dulu...")
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics", "onnx", "onnxslim"], check=True)
    from ultralytics import YOLO

import pathlib

# Pakai data.yaml gabungan (Opsi C) kalau ada; kalau tidak, fallback ke data.yaml manual (sel 8).
_merged = pathlib.Path("/kaggle/working/pharmacy_merged/data.yaml")
DATA_YAML = str(_merged) if _merged.exists() else "/kaggle/working/data.yaml"
print("Training dengan:", DATA_YAML)

model = YOLO("yolo26n.pt")

results = model.train(
    data=DATA_YAML,
    epochs=120,
    imgsz=640,
    batch=16,          # T4 Kaggle sanggup batch lebih besar drpd MX350 laptop (batch=4-8)
    device=0,
    patience=30,
    workers=4,
    project="/kaggle/working/runs",
    name="medicine_yolo26n",
)


## 6. Validasi hasil

Cek mAP50 / mAP50-95 per kelas. Jika salah satu SKU jauh lebih rendah dari yang lain,
biasanya artinya SKU itu kurang contoh foto atau variasi sudut/pencahayaan di dataset —
tambah foto untuk SKU tersebut dan latih ulang.


In [ ]:
metrics = model.val()
print(metrics.box.maps)     # mAP50-95 per kelas
print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)


## 7. Export ke ONNX (format yang dipakai `src/detector.py` di laptop)

`opset=12` dipilih untuk kompatibilitas luas dengan `onnxruntime` versi lama sekalipun
(relevan karena laptop memakai CUDA 12.6/12.8 + PyTorch/onnxruntime yang dipin ke versi
yang masih mendukung Pascal/MX350 — lihat dokumen desain bagian 3).


In [ ]:
best_weights = "/kaggle/working/runs/medicine_yolo26n/weights/best.pt"
best_model = YOLO(best_weights)
onnx_path = best_model.export(format="onnx", imgsz=640, opset=12, simplify=True)
print("ONNX tersimpan di:", onnx_path)


## 8. Bungkus hasil untuk di-download ke laptop

Setelah sel ini jalan, buka panel **Output** di Kaggle (kanan bawah / ikon folder),
lalu download `medicine_model_export.zip`.

Langkah di laptop Anda setelah download:
```bash
unzip medicine_model_export.zip -d ~/Downloads/medicine_model_export
cp ~/Downloads/medicine_model_export/best.onnx  medicine_shelf_vision/models/medicine_yolo26n.onnx
cp ~/Downloads/medicine_model_export/classes.txt medicine_shelf_vision/data/classes.txt

# uji cepat dengan webcam:
python scripts/test_webcam.py --onnx models/medicine_yolo26n.onnx --classes data/classes.txt

# uji pencarian tingkat rak dengan foto rak sungguhan:
python scripts/run_shelf_search.py --target MED-001 --live --image data/test_assets/foto_rak_saya.jpg
```

(Ingat: SKU/kelas di `data/classes.txt` harus disamakan urutannya dengan `data.yaml` training,
dan `medicine_db.sqlite` — lihat `scripts/init_db.py` — harus diisi ulang dengan SKU nyata Anda.)


In [ ]:
import shutil, os

export_dir = "/kaggle/working/medicine_model_export"
os.makedirs(export_dir, exist_ok=True)

shutil.copy(onnx_path, os.path.join(export_dir, "best.onnx"))
shutil.copy(best_weights, os.path.join(export_dir, "best.pt"))

# tulis daftar nama kelas dalam urutan index yang sama dengan data.yaml
with open(os.path.join(export_dir, "classes.txt"), "w") as f:
    for idx, name in model.names.items():
        f.write(name + "\n")

shutil.make_archive("/kaggle/working/medicine_model_export", "zip", export_dir)
print("Selesai. Download medicine_model_export.zip dari panel Output Kaggle.")
